### Q1:
#### (a):

In [99]:
import pandas as pd
import numpy as np
import re
from collections import Counter, defaultdict

def load_data(path):
    try:
        df = pd.read_csv(path)
        required_cols = ['review_text', 'category']
        for col in required_cols:
            if col not in df.columns:
                raise ValueError(f"Missing column: {col}")
        return df
    except Exception as e:
        print("Error loading file:", e)

stopwords = set([
 'is', 'and', 'in', 'to', 'of', 'this', 'that', 'it', 'for', 'on', 'with', 'as', 'was', 'but'
])

def preprocess(text):
    text = str(text)
    text = text.lower()
    text = re.sub(r'<.*?>', ' ', text)
    text = re.sub(r'[^a-z\s]', '', text)
    text = re.sub(r'\s+', ' ', text)
    tokens = text.strip().split()
    tokens = [word for word in tokens if word not in stopwords]
    return tokens

df = load_data("shopsense_reviews.csv")
df['review_text'] = df['review_text'].fillna('')
df['tokens'] = df['review_text'].apply(preprocess)

In [100]:
print(len(df['tokens']))

10199


In [101]:
print(sum(1 for doc in df['tokens'] if "fabric" in doc))

0


In [102]:
def build_vocab(docs):
    vocab = {}
    idx = 0
    for doc in docs:
        for word in doc:
            if word not in vocab:
                vocab[word] = idx
                idx += 1
    return vocab

vocab = build_vocab(df['tokens'])

In [103]:
vocab

{'do': 0,
 'not': 1,
 'buy': 2,
 'fake': 3,
 'product': 4,
 'nothing': 5,
 'like': 6,
 'the': 7,
 'images': 8,
 'shown': 9,
 'website': 10,
 'waste': 11,
 'money': 12,
 'too': 13,
 'many': 14,
 'issues': 15,
 'very': 16,
 'poor': 17,
 'quality': 18,
 'control': 19,
 'loved': 20,
 'perfect': 21,
 'daily': 22,
 'use': 23,
 'my': 24,
 'whole': 25,
 'family': 26,
 'loves': 27,
 'smartphones': 28,
 'amazing': 29,
 'value': 30,
 'technical': 31,
 'exceeded': 32,
 'expectations': 33,
 'every': 34,
 'way': 35,
 'possible': 36,
 'good': 37,
 'using': 38,
 'since': 39,
 'weeks': 40,
 'no': 41,
 'complaints': 42,
 'whatsoever': 43,
 'recommended': 44,
 'bahut': 45,
 'accha': 46,
 'hai': 47,
 'top': 48,
 'notch': 49,
 'will': 50,
 'again': 51,
 'sure': 52,
 'received': 53,
 'a': 54,
 'defective': 55,
 'kitchen': 56,
 'asked': 57,
 'replacement': 58,
 'still': 59,
 'waiting': 60,
 'after': 61,
 'excellent': 62,
 'satisfied': 63,
 'would': 64,
 'definitely': 65,
 'recommend': 66,
 'friends': 67,
 'b

In [104]:
def compute_tf(doc, vocab):
    tf = np.zeros(len(vocab))
    word_counts = Counter(doc)
    total_words = len(doc)
    
    for word, count in word_counts.items():
        if word in vocab:
            tf[vocab[word]] = count / total_words
    return tf

In [105]:
def compute_idf(docs, vocab):
    N = len(docs)
    idf = np.zeros(len(vocab))
    
    for word, idx in vocab.items():
        df_count = sum(1 for doc in docs if word in doc)
        idf[idx] = np.log(N / (1 + df_count))
    
    return idf

In [106]:
def compute_tfidf(docs, vocab):
    idf = compute_idf(docs, vocab)
    tfidf_matrix = []
    
    for doc in docs:
        tf = compute_tf(doc, vocab)
        tfidf = tf * idf
        tfidf_matrix.append(tfidf)
    
    return np.array(tfidf_matrix), idf

tfidf_matrix, idf = compute_tfidf(df['tokens'], vocab)

In [107]:
tfidf_matrix

array([[0.2737187 , 0.19138516, 0.19327451, ..., 0.        , 0.        ,
        0.        ],
       [0.        , 0.        , 0.        , ..., 0.        , 0.        ,
        0.        ],
       [0.2737187 , 0.19138516, 0.19327451, ..., 0.        , 0.        ,
        0.        ],
       ...,
       [0.        , 0.        , 0.        , ..., 0.        , 0.        ,
        0.        ],
       [0.        , 0.        , 0.        , ..., 0.        , 0.        ,
        0.        ],
       [0.        , 0.        , 0.        , ..., 0.        , 0.        ,
        0.        ]], shape=(10199, 238))

In [108]:
idf

array([3.28462435, 2.29662193, 2.31929417, 4.08255048, 0.93624535,
       2.95905652, 4.08255048, 1.22601345, 4.08255048, 4.08255048,
       4.08255048, 3.30311893, 2.33436226, 2.59804318, 4.12409948,
       4.12409948, 1.57512791, 3.45239263, 1.10323123, 4.12409948,
       2.86185777, 2.86185777, 2.62610113, 2.62610113, 1.75697587,
       2.86185777, 2.15677524, 2.86185777, 4.54791373, 2.49189246,
       2.81005003, 4.26720032, 2.81005003, 2.81005003, 2.13332358,
       2.81005003, 2.81005003, 2.40025122, 2.88616452, 2.88616452,
       2.61130597, 2.88616452, 2.88616452, 2.88616452, 2.76234623,
       2.67611155, 2.46963026, 2.10960058, 2.79710486, 2.79710486,
       2.79710486, 2.79710486, 2.79710486, 4.03154792, 4.03154792,
       4.03154792, 4.71918545, 4.03154792, 4.03154792, 4.03154792,
       4.03154792, 2.96474374, 2.8364542 , 2.17214702, 2.8364542 ,
       2.8364542 , 2.8364542 , 2.8364542 , 2.82978751, 1.64121508,
       2.82978751, 2.82978751, 2.82978751, 2.07386832, 1.85604

#### (b):

In [109]:
import numpy as np

def cosine_similarity(vec1, vec2):
    dot_product = 0
    norm1 = 0
    norm2 = 0
    
    for i in range(len(vec1)):
        dot_product += vec1[i] * vec2[i]
        norm1 += vec1[i] ** 2
        norm2 += vec2[i] ** 2
    
    norm1 = norm1 ** 0.5
    norm2 = norm2 ** 0.5
    
    if norm1 == 0 or norm2 == 0:
        return 0
    
    return dot_product / (norm1 * norm2)

In [110]:
def query_ranking(query, df, vocab, idf, tfidf_matrix):
    query_tokens = preprocess(query)
    query_tf = compute_tf(query_tokens, vocab)
    query_vec = query_tf * idf
    
    scores = []
    
    for i in range(len(tfidf_matrix)):
        score = cosine_similarity(query_vec, tfidf_matrix[i])
        scores.append((i, score))
    
    # Sort by similarity
    scores.sort(key=lambda x: x[1], reverse=True)
    
    # Ensure unique reviews
    seen = set()
    top_results = []
    
    for idx, score in scores:
        review = df.iloc[idx]['review_text']
        
        if review not in seen:
            top_results.append((idx, score))
            seen.add(review)
        
        if len(top_results) == 5:
            break
    
    return top_results

In [111]:
query_ranking('wireless earbuds battery life poor',df,vocab,idf,tfidf_matrix)

[(637, np.float64(0.3403534869838122)),
 (125, np.float64(0.29981373983913223)),
 (502, np.float64(0.28827114173467866)),
 (52, np.float64(0.27222057615080814)),
 (55, np.float64(0.23435003644519248))]

In [112]:
top5 = query_ranking("wireless earbuds battery life poor", df, vocab, idf, tfidf_matrix)

for idx, score in top5:
    print("Score:", score)
    print("Review:", df.iloc[idx]['review_text'])
    print("-" * 80)

Score: 0.3403534869838122
Review: Superb! The earbuds is exactly as described. Very happy with this purchase.
--------------------------------------------------------------------------------
Score: 0.29981373983913223
Review: Loved it!! Perfect for daily use. My whole family loves this earbuds.
--------------------------------------------------------------------------------
Score: 0.28827114173467866
Review: Amazing value for money. The earbuds exceeded my expectations in every way possible.
--------------------------------------------------------------------------------
Score: 0.27222057615080814
Review: Outstanding quality. I was skeptical at first but this earbuds proved me wrong completely.
--------------------------------------------------------------------------------
Score: 0.23435003644519248
Review: Bohot accha product hai ye! Camera quality is amazing. Battery backup bhi theek hai. Recommended!
--------------------------------------------------------------------------------


#### (c):

In [113]:
from sklearn.feature_extraction.text import TfidfVectorizer

def sklearn_compare(df):
    corpus = df['review_text'].values

    vectorizer = TfidfVectorizer()
    sk_tfidf = vectorizer.fit_transform(corpus).toarray()
    
    min_shape = min(tfidf_matrix.shape[1], sk_tfidf.shape[1])
    
    diff = np.linalg.norm(tfidf_matrix[:, :min_shape] - sk_tfidf[:, :min_shape])
    avg_diff = diff / tfidf_matrix.shape[0]
    
    return avg_diff

print("Average L2 Difference:", sklearn_compare(df))

Average L2 Difference: 0.012052866230359872


#### (d):

In [114]:
def top_word_category(df, tfidf_matrix, vocab, category_name, idf, min_idf=2):
    indices = df[df['category'] == category_name].index
    subset = tfidf_matrix[indices]
    
    avg_scores = np.mean(subset, axis=0)
    
    # Apply IDF filter
    avg_scores = np.where(idf > min_idf, avg_scores, 0)
    
    inv_vocab = {v:k for k,v in vocab.items()}
    
    top_idx = np.argmax(avg_scores)
    return inv_vocab[top_idx]

In [115]:
top_word_category(df, tfidf_matrix, vocab, 'Electronics', idf, min_idf=2)

'cameras'

#### "Cameras" ranks highest because it is frequently used in Electronics reviews but not common across all categories, resulting in a high TF-IDF score and making it a strong category-specific term.

### Q2:
#### (a):

The word "fabric" does not appear in the dataset after preprocessing, resulting in TF = 0 and DF = 0. Although the IDF is high, the TF-IDF score remains zero because the term is absent in the document.

In [116]:
import numpy as np

doc_index = 41
doc_tokens = df.iloc[doc_index]['tokens']
all_docs = df['tokens'].tolist()

word = "fabric"

# TF
tf = doc_tokens.count(word) / len(doc_tokens) if len(doc_tokens) > 0 else 0

df_count = sum(1 for doc in all_docs if word in doc)
N = len(all_docs)
idf = np.log(N / (1+df_count))

# TF-IDF
tfidf = tf * idf

print("TF:", tf)
print("DF:", df_count)
print("IDF:", idf)
print("TF-IDF:", tfidf)

TF: 0.0
DF: 0
IDF: 9.230044955250518
TF-IDF: 0.0


In [117]:
doc_index = 41  # Doc_42 (0-based indexing)
doc_tokens = df.iloc[doc_index]['tokens']

total_words = len(doc_tokens)

print("Total words in Doc_42:", total_words)

Total words in Doc_42: 11


#### Term Frequency (TF):
### TF(fabric,Doc42​) = Count of "fabric" in Doc42​ ​/ Total words in Doc42
 =0/11 = 0
#### Document Frequency (DF):
DF(fabric)=0
#### Inverse Document Frequency (IDF)
Theoretical formula:
### IDF = ln (N/DF)
Since DF=0, this becomes undefined.
#### Therefore, we use smoothed IDF:
### IDF = ln(N/(1+DF))
IDF = ln(10199/(1+0))
IDF = 9.23
#### TF-IDF = TF*IDF
TF-IDF = 0

Since the word "fabric" does not appear in Doc_42, its TF is zero, resulting in a TF-IDF score of zero despite having a high IDF. This demonstrates that TF-IDF depends on both term presence and rarity.

#### (b):

In [119]:
import numpy as np

all_docs = df['tokens'].tolist()
N = len(all_docs)

words = ["the", "embroidery"]

for word in words:
    df_count = sum(1 for doc in all_docs if word in doc)
    idf = np.log(N / (1 + df_count)) 
    
    print(f"Word: {word}")
    print(f"DF: {df_count}")
    print(f"IDF: {idf}")
    print("-" * 40)

Word: the
DF: 2992
IDF: 1.2260134473978193
----------------------------------------
Word: embroidery
DF: 0
IDF: 9.230044955250518
----------------------------------------


### IDF('the') = ln(N / (1 + DF_the)) 
= ln(10000 / (1 + 2992)) ≈ 1
### IDF('embroidery') = ln(N / (1 + DF_embroidery)) 
= ln(10000 / (1 + 0)) ≈ 9.21

The word "the" appears in many documents, so its document frequency is high, making its IDF value close to 0. In contrast, "embroidery" appears in no document, resulting in a low document frequency and a high IDF value, which makes it more informative.

#### (c):

Using only word frequency is not sufficient because it treats all words as equally important, even very common words like "the", "is", and "and". These words appear frequently in almost all documents but do not provide any meaningful information about the content.

TF-IDF improves this by reducing the importance of such common words using IDF, while giving higher importance to rare and informative words like "embroidery" or "battery". This helps in better understanding the actual meaning of a document and improves tasks like search, ranking, and classification.

Therefore, TF-IDF is not overcomplicated but actually more effective, as it balances both frequency and importance to identify truly meaningful words.